# 01. Data and Table 2

Districts used in the analysis and the yearly summary (manuscript Table 2).
Run `python scripts/run_pipeline.py` first (or the processed tables are built on demand).

In [ ]:
from tbopt import config, data
import pandas as pd

## Filtering

Kept districts: `h > 0`, `D > 0`, and reconstructed `D_t > 0`, `N_t > 0` for every 40+ age group.

In [ ]:
counts = pd.DataFrame({
    y: {"all": len(data.load_district(y)), "kept": len(data.load_processed(y))}
    for y in config.YEARS
}).T
counts

## Table 2

In [ ]:
table2 = data.summary_table()
table2.style.format({'D_40': '{:.1f}'})

## Check: processed data equals the Dryad filtered files

In [ ]:
for y in config.YEARS:
    p, f = data.load_processed(y), data.load_district(y, filtered=True)
    assert len(p) == len(f) and all(p[c].sum() == f[c].sum() for c in ["N", "D", "h"]), y
print("OK")

## Check: province age fractions vs. province count tables

The district files carry province age fractions `n_t`, `d_t`. Recomputing them from the Dryad province count tables (`sido_*_age_2014_2022.csv`) gives the same values except for the cases listed below (to be verified against the KDCA annual reports).

In [ ]:
import numpy as np
rows = []
for y in config.YEARS:
    for kind, used, recomputed in zip("nd", data.sido_age_fractions(y), data.sido_age_fractions_from_counts(y)):
        diff = (used - recomputed.loc[used.index]).abs()
        for s in diff.index[(diff > 1e-6).any(axis=1)]:
            rows.append({"year": y, "fraction": kind, "sido": s, "max_abs_diff": float(diff.loc[s].max())})
pd.DataFrame(rows)